<a href="https://colab.research.google.com/github/Dat1205/Deepfake-Detection-/blob/main/DEEPFAKE_PROJECT_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **LƯU Ý:** trước khi run code, luôn đảm bảo T4 GPU đang được sử dụng để cho tốc độ và hiệu năng xử lý hình ảnh tốt nhất bằng cách vào menu `Runtime` -> `Change Runtime Type` -> mục `Hardware Accelerator` chọn `T4 GPU`


> **LƯU Ý:** Nếu bạn muốn chạy và đánh giá nhiều mô hình trong một phiên làm việc, trước khi khởi chạy mô hình (hoặc mô hình baseline), hãy làm sạch môi trường Colab bằng cách chọn menu `Runtime -> Restart Session`. Lệnh này sẽ xóa mọi biến tạm của model học máy (như biến `model`, `x_train`, `y_pred`), hạn chế sai lệch hoặc ảnh hưởng tới kết quả của những mô hình được chạy sau.

> Việc này không xóa những dữ liệu bạn đã tải từ máy lên (ví dụ như thư mục ảnh `Processed_Data`) và những dữ liệu đã được lưu trong ổ cứng của runtime, nên bạn không cần phải thực hiện lại thao tác tải lên nếu không cần thiết.

> Tuy nhiên, bạn sẽ cần phải chạy lại các lệnh import (bước 1 và bước khởi tạo mô hình, hoặc bước khởi tạo baseline)




# Bước 1: Cài đặt và import các thư viện

In [ ]:
!pip install kaggle timm facenet-pytorch opencv-python matplotlib scikit-learn tqdm --quiet --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 24.1 MB/s eta 0:00:00


In [ ]:
import os
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# QUAN TRỌNG: import các mô hình trong đó có VGG16 (vgg) --------------------
from torchvision import models
# ---------------------------------------------------------------------------

import timm
from facenet_pytorch import MTCNN
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import gc # Garbage Collector

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Bước 2: Tải dataset

### **2.0. (TÙY CHỌN) Tải lên ảnh đã được trích xuất từ trước**
Link tải:
- Độ phân giải 224x224 (cho EfficientNetB0, Diffusion):
  - [deepfake-detection-challenge-preview](https://drive.google.com/file/d/1qMLIpawXzxoSh5nw-F6knQ3qflhLiaYt/view?usp=sharing)
  - [FaceForensics++-C23-full](https://drive.google.com/open?id=1zJHmvbfNhD2bEGiLd35Mxr8egaEev2T3&usp=drive_fs)
- Độ phân giải 299x299 (cho Xception):
  - [deepfake-detection-challenge-preview](https://drive.google.com/file/d/1IqR1bVMI9het74paEnt2r7XxwR1Eecil/view?usp=sharing)
  - [FaceForensics++-C23-full](https://drive.google.com/file/d/1MiQcW26BAlCs1g6O8sFC6FdD_IRPALSU/view?usp=sharing)


Đầu tiên, hãy tải tập ảnh (file nén định dạng .zip) về máy.

Sau đó chạy cell code bên dưới. Bấm chọn tải lên file vừa mới download.

Sau bước này, ảnh đã cắt và phân loại FAKE/REAL sẽ được lưu ở `/content/Processed_Data`.

Nếu đã tải lên ảnh được cắt sẵn, thì hãy ***chuyển luôn sang chạy bước 4 hoặc BASELINE*** để thực hiện huấn luyện mô hình. Bạn muốn chạy mô hình/baseline nào thì mở mục lục (Table of Contents) (bên trái màn hình Collab) và bấm vào mô hình đó.

In [ ]:
from google.colab import files
files.upload()
!unzip -q *.zip -d /

## 2.1. Option 1: Deepfake-detection-challenge-preview
Để tải dataset này, ta cần sử dụng Kaggle API để tải dataset từ contest của Kaggle


> *Để sử dụng Kaggle API, bạn cần có khóa API (`kaggle.json`). Nếu bạn chưa có, hãy thực hiện các bước sau:*

> *1. Truy cập trang web Kaggle và đăng nhập.*

> *2. Đi tới hồ sơ của bạn (Your Profile) -> Tài khoản (Account).*

> *3. Cuộn xuống phần "API" và nhấp vào "Create New API Token". Thao tác này sẽ tải xuống tệp `kaggle.json` chứa tên người dùng và khóa API của bạn.*

> *Sau đó, bạn cần tải tệp `kaggle.json` này lên môi trường Colab của mình. Một cách để làm điều này là tải nó lên thư mục gốc của phiên hiện tại hoặc vào thư mục `.kaggle`:*

Lưu ý rằng, việc tải tệp `kaggle.json` **cần được thực hiện mỗi khi bắt đầu 1 phiên làm việc mới** với runtime mới (do tệp này cùng các dữ liệu làm việc sẽ bị xoá theo runtime cũ khi phiên làm việc cũ kết thúc).

Sử dụng lệnh Kaggle API đã cung cấp để tải dataset từ cuộc thi `deepfake-detection-challenge`:

***Lưu ý:***
Nếu chưa tham gia cuộc thi, khi chạy lệnh này bạn sẽ gặp lỗi `403 - Forbidden when download dataset`

Trước khi tải xuống, phải vào liên kết cuộc thi và bấm "Join the competition". Truy cập đến cuộc thi nằm tại [**đường link này**](https://www.kaggle.com/competitions/deepfake-detection-challenge/data).

In [ ]:
# Tải tệp kaggle.json lên môi trường Colab bằng hộp thoại Upload
from google.colab import files
files.upload()

# Tạo thư mục .kaggle và di chuyển tệp kaggle.json vào đó
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/

# Đặt quyền cho tệp kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

# Kiểm tra sự tồn tại của file kaggle.json
!ls -la ~/.kaggle

In [ ]:
# Tải dataset từ cuộc thi 'deepfake-detection-challenge'
!kaggle competitions download -c deepfake-detection-challenge
# Kiểm tra file dataset được tải về
!ls -la ./

Giải nén file dataset vừa tải về

In [ ]:
# Tạo thư mục lưu trữ dataset
!mkdir -p data/DFDC

# Giải nén
!unzip -q deepfake-detection-challenge.zip -d data/DFDC

# Kiểm tra dataset vừa được giải nén
! ls -la data/DFDC/

## 2.2. Option 2: Celeb-DF v2

In [ ]:
# Tải trực tiếp dataset bằng thư viện kagglehub (không cần API)
import kagglehub
reubensuju_celeb_df_v2_path = kagglehub.dataset_download('reubensuju/celeb-df-v2')
print(f'Celeb-df-v2 đã đuợc tải xuống tại {reubensuju_celeb_df_v2_path}')

## 2.3. Option 3. FaceForensics++

In [ ]:
# Tải bộ dataset FaceForensics++
import kagglehub

# Download latest version
ff_path = kagglehub.dataset_download("xdxd003/ff-c23")
print(f'FaceForensics++ đã được tải xuống tại {ff_path}')

100%|██████████| 16.7G/16.7G [03:20<00:00, 89.1MB/s]

Extracting files...


FaceForensics++ đã được tải xuống tại /root/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1


# Bước 3: Trích xuất khuôn mặt phục vụ cho việc huấn luyện
Bước này sẽ đọc video, dùng thư viện facenet-pytorch (bạn đã cài) để cắt khuôn mặt, và lưu ảnh vào Google Drive để dùng lại cho cả 2 mô hình sau này.

Mục tiêu: Tạo thư mục Processed_Data trên Drive gồm tập Train và Validation.

Phương pháp: trích xuất khuôn mặt kiểu VGG16 (cứ 10 frame thì lấy 1 frame)

## 3.1. Dành cho Deepfake-Detection-Challenge dataset

In [ ]:
# --- CODE PHẦN 1: TRÍCH XUẤT KHUÔN MẶT RA THƯ MỤC MỚI (KHÔNG GHI ĐÈ) ---
import os
import json
import cv2
import torch
import numpy as np
from facenet_pytorch import MTCNN
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# 1. Cấu hình đường dẫn
DATA_ROOT = 'data/DFDC' # Đường dẫn chứa dữ liệu video gốc
VIDEO_DIR = os.path.join(DATA_ROOT, 'train_sample_videos')
METADATA_PATH = os.path.join(VIDEO_DIR, 'metadata.json')

# --- THAY ĐỔI: Đặt tên thư mục mới để không ghi đè thư mục cũ ---
DRIVE_SAVE_PATH = '/content/Processed_Data'
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)
print(f"Dữ liệu sẽ được lưu tại thư mục MỚI: {DRIVE_SAVE_PATH}")

# 2. Thiết lập MTCNN
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
####################### Chỉnh lại Image_size = 299 (Xception), 224 (EfficientNetB0, Diffusion) TRƯỚC KHI CẮT ##############
mtcnn = MTCNN(image_size=224, margin=10, keep_all=False, select_largest=True, post_process=False, device=device)
###########################################################################################################################

# 3. Đọc Metadata và chia tập Train/Val
with open(METADATA_PATH, 'r') as f:
    metadata = json.load(f)

video_files = []
labels = []
for filename, data in metadata.items():
    if os.path.exists(os.path.join(VIDEO_DIR, filename)):
        video_files.append(filename)
        labels.append(1 if data['label'] == 'FAKE' else 0)

# Chia tập dữ liệu: 80% Train, 20% Val
train_videos, val_videos, train_labels, val_labels = train_test_split(
    video_files, labels, test_size=0.2, random_state=42, stratify=labels
)

# 4. Hàm xử lý video thành ảnh (CƠ CHẾ STEP)
def process_dataset_new_folder(video_list, data_labels, mode='train', step=10):
    save_dir = os.path.join(DRIVE_SAVE_PATH, mode)

    # Tạo thư mục con
    os.makedirs(os.path.join(save_dir, 'REAL'), exist_ok=True)
    os.makedirs(os.path.join(save_dir, 'FAKE'), exist_ok=True)

    print(f"Đang xử lý tập {mode} (lưu vào folder mới)...")

    for vid_name, label in tqdm(zip(video_list, data_labels), total=len(video_list)):
        label_str = 'FAKE' if label == 1 else 'REAL'
        video_path = os.path.join(VIDEO_DIR, vid_name)

        cap = cv2.VideoCapture(video_path)

        frame_idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            # --- Chỉ lấy frame nếu chia hết cho step ---
            if frame_idx % step == 0:
                try:
                    # Chuyển BGR sang RGB
                    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                    # Phát hiện và crop khuôn mặt
                    face = mtcnn(frame_rgb, save_path=None)

                    if face is not None:
                        # Chuyển tensor về ảnh numpy
                        face_img = face.permute(1, 2, 0).int().numpy().astype(np.uint8)

                        # Lưu ảnh
                        save_name = f"{vid_name.split('.')[0]}_{frame_idx}.jpg"
                        save_path = os.path.join(save_dir, label_str, save_name)
                        cv2.imwrite(save_path, cv2.cvtColor(face_img, cv2.COLOR_RGB2BGR))
                except Exception as e:
                    pass

            frame_idx += 1

        cap.release()

# 5. Thực thi
# step=10: Cứ 10 frame lấy 1 frame
process_dataset_new_folder(train_videos, train_labels, mode='train', step=10)
process_dataset_new_folder(val_videos, val_labels, mode='val', step=10)

print(f"Hoàn tất! Kiểm tra thư mục: {DRIVE_SAVE_PATH}")

## 3.2. Dành cho Celeb-DF v2

In [ ]:
# 1. Cấu hình đường dẫn
from pathlib import Path
CELEB_ROOT = Path(reubensuju_celeb_df_v2_path)
DRIVE_SAVE_PATH = Path('/content/Extracted_Data')
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)
print(f"Dữ liệu sẽ được lưu tại thư mục MỚI: {DRIVE_SAVE_PATH}")

In [ ]:
# 2. Định nghĩa hàm cắt khuôn mặt (tương tự FaceForensics++)
import os
import torch
import numpy as np
from facenet_pytorch import MTCNN
from pathlib import Path
from tqdm import tqdm
import cv2

# Import Decord
try:
    from decord import VideoReader, cpu, gpu
    DECORD_AVAILABLE = True
except ImportError:
    print("⚠️  Decord not installed. Install with: pip install decord")
    DECORD_AVAILABLE = False

# ==================== GOOGLE COLAB SINGLE GPU PROCESSOR WITH DECORD ====================

class ColabSingleGPUProcessor:
    """
    Xử lý video với Decord trên Google Colab (Single GPU)
    Tối ưu cho T4/V100/A100 GPU
    """

    def __init__(self, batch_size=32, use_gpu_decode=False):
        """
        Args:
            batch_size: Số frames xử lý cùng lúc trên GPU
            use_gpu_decode: True = GPU decode (cần build Decord), False = CPU decode
        """
        if not DECORD_AVAILABLE:
            raise RuntimeError("Decord is required. Install: pip install decord")

        print("Initializing Google Colab Single GPU Processor...")
        print(f"PyTorch version: {torch.__version__}")
        print(f"CUDA available: {torch.cuda.is_available()}")

        if not torch.cuda.is_available():
            raise RuntimeError("No GPU available! Please enable GPU in Colab: Runtime > Change runtime type > GPU")

        # In thông tin GPU
        props = torch.cuda.get_device_properties(0)
        vram_gb = props.total_memory / 1e9
        print(f"GPU: {props.name} ({vram_gb:.1f}GB VRAM)")

        self.batch_size = batch_size
        self.device = 'cuda:0'

        # Kiểm tra Decord GPU support
        try:
            from decord import bridge
            has_gpu = bridge.bridge_available()
            print(f"\nDecord GPU support: {has_gpu}")
        except:
            has_gpu = False
            print(f"\nDecord GPU support: {has_gpu}")

        # Chọn context cho Decord
        if use_gpu_decode and has_gpu:
            print("✓ Using GPU hardware decoding (NVDEC)")
            self.decord_ctx = gpu(0)
            self.use_gpu_decode = True
        else:
            if use_gpu_decode and not has_gpu:
                print("⚠️  GPU decode requested but not available, falling back to CPU")
            else:
                print("✓ Using CPU decoding")
            self.decord_ctx = cpu(0)
            self.use_gpu_decode = False

        # Khởi tạo MTCNN model
        print("\nInitializing MTCNN model...")
        self.mtcnn = MTCNN(
            image_size=224,
            margin=0,
            min_face_size=20,
            thresholds=[0.6, 0.7, 0.7],
            factor=0.709,
            post_process=True,
            device=self.device,
            keep_all=False
        )
        print("  ✓ MTCNN model ready")

    def process_single_video(self, video_path, output_folder, sample_rate=5):
        """
        Xử lý 1 video với Decord

        Args:
            video_path: Đường dẫn video
            output_folder: Thư mục lưu frames
            sample_rate: Lấy 1 frame mỗi N frames
        """
        output_folder = Path(output_folder)
        output_folder.mkdir(parents=True, exist_ok=True)

        try:
            # Mở video với Decord
            vr = VideoReader(str(video_path), ctx=self.decord_ctx)
            total_frames = len(vr)

            # Tính indices cần lấy
            frame_indices = list(range(0, total_frames, sample_rate))

            if not frame_indices:
                return 0

            saved_count = 0

            # Xử lý theo batch
            for i in range(0, len(frame_indices), self.batch_size):
                batch_idx = frame_indices[i:i + self.batch_size]

                # Load batch frames với Decord (nhanh hơn cv2 nhiều)
                frames_batch = vr.get_batch(batch_idx).asnumpy()

                # Convert thành list
                frame_list = [frames_batch[j] for j in range(len(frames_batch))]

                # Detect faces với MTCNN
                try:
                    with torch.no_grad():
                        boxes, probs = self.mtcnn.detect(frame_list)

                    # Lưu faces
                    for frame, box, prob, idx in zip(frame_list, boxes, probs, batch_idx):
                        if box is not None and prob is not None and prob[0] > 0.9:
                            x1, y1, x2, y2 = [int(b) for b in box[0]]
                            face = frame[y1:y2, x1:x2]

                            if face.size > 0:
                                # Decord trả về RGB, convert sang BGR cho cv2.imwrite
                                face_bgr = cv2.cvtColor(face, cv2.COLOR_RGB2BGR)
                                output_path = output_folder / f"frame_{idx:06d}.jpg"
                                cv2.imwrite(str(output_path), face_bgr)
                                saved_count += 1
                except:
                    pass  # Silent error handling

            return saved_count

        except Exception as e:
            return 0

    def process_videos(self, video_paths, output_base, sample_rate=5):
        """
        Xử lý danh sách videos tuần tự

        Args:
            video_paths: List các đường dẫn video
            output_base: Thư mục output
            sample_rate: Lấy 1 frame mỗi N frames
        """
        output_base = Path(output_base)
        results = {}

        for video_path in tqdm(video_paths, desc="Processing videos", unit="video", leave=True, position=0):
            output_folder = output_base / video_path.stem

            count = self.process_single_video(
                video_path, output_folder, sample_rate
            )

            results[video_path.name] = count


        return results

    def cleanup(self):
        """
        Làm sạch VRAM và giải phóng tài nguyên
        """
        print("\nCleaning up GPU memory...")

        # Xóa cache của PyTorch
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

        # Garbage collection
        import gc
        gc.collect()

        print("✓ GPU memory cleaned")

        print(f"{'='*60}\n")

In [ ]:
# 3. Trích xuất chân dung từ video trong dataset Celeb-DF v2
def process_celebdf():
    real_out = DRIVE_SAVE_PATH / 'real'
    fake_out = DRIVE_SAVE_PATH / 'fake'
    print("\n=== Celeb-DF REAL ===")
    real_path = CELEB_ROOT / "Celeb-real"
    real_videos = list(real_path.glob("*.mp4"))
    print(f"Found {len(real_videos)} real videos")
    results_real = processor.process_videos(
      real_videos,
      real_out,
      sample_rate=10
    )
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    print("\n=== Youtube-real ===")
    youtube_real_path = CELEB_ROOT / "YouTube-real"
    youtube_real_videos = list(youtube_real_path.glob("*.mp4"))
    print(f"Found {len(youtube_real_videos)} YouTube real videos")
    results_youtube_real = processor.process_videos(
          youtube_real_videos,
          real_out,
          sample_rate=10
    )
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    print("\n=== Celeb-DF FAKE ===")
    fake_path = CELEB_ROOT / "Celeb-synthesis"
    fake_videos = list(fake_path.glob("*.mp4"))
    print(f"Found {len(fake_videos)} fake videos")
    result_fake = processor.process_videos(
      fake_videos,
      fake_out,
      sample_rate=10
    )
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

processor = ColabSingleGPUProcessor(batch_size=32, use_gpu_decode=False)
process_celebdf()
print(f"Hoàn tất! Kiểm tra thư mục: {DRIVE_SAVE_PATH}")

In [ ]:
import os
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import json
import random

# ==================== DATASET BUILDER ====================

class DeepfakeDatasetBuilder:
    """
    Xây dựng dataset train/val từ thư mục extracted frames
    """

    def __init__(self, extracted_path, output_path, train_ratio=0.8, random_seed=42):
        """
        Args:
            extracted_path: Thư mục chứa real/ và fake/ folders
            output_path: Thư mục output cho dataset
            train_ratio: Tỉ lệ train (0.8 = 80% train, 20% val)
            random_seed: Random seed để reproducible
        """
        self.extracted_path = Path(extracted_path)
        self.output_path = Path(output_path)
        self.train_ratio = train_ratio
        self.random_seed = random_seed

        # Set random seed
        random.seed(random_seed)

        # Tạo cấu trúc thư mục
        self.train_path = self.output_path / 'train'
        self.val_path = self.output_path / 'val'

        self.train_real = self.train_path / 'real'
        self.train_fake = self.train_path / 'fake'
        self.val_real = self.val_path / 'real'
        self.val_fake = self.val_path / 'fake'

        print(f"Dataset Builder initialized:")
        print(f"  Source: {self.extracted_path}")
        print(f"  Output: {self.output_path}")
        print(f"  Train ratio: {train_ratio*100:.0f}%")
        print(f"  Val ratio: {(1-train_ratio)*100:.0f}%")

    def build_dataset(self):
        """
        Xây dựng dataset từ extracted frames
        """
        print(f"\n{'='*60}")
        print("BUILDING DATASET")
        print(f"{'='*60}\n")

        # 1. Thu thập tất cả video folders
        print("Step 1: Collecting video folders...")
        real_folders, fake_folders = self._collect_video_folders()

        print(f"  Found {len(real_folders)} real video folders")
        print(f"  Found {len(fake_folders)} fake video folders")

        # 2. Split train/val theo video-level (không phải frame-level)
        print("\nStep 2: Splitting train/val...")
        train_real, val_real = self._split_folders(real_folders, self.train_ratio)
        train_fake, val_fake = self._split_folders(fake_folders, self.train_ratio)

        print(f"  Train: {len(train_real)} real + {len(train_fake)} fake videos")
        print(f"  Val:   {len(val_real)} real + {len(val_fake)} fake videos")

        # 3. Tạo cấu trúc thư mục
        print("\nStep 3: Creating directory structure...")
        self._create_directory_structure()

        # 4. Copy frames vào train/val folders
        print("\nStep 4: Copying frames...")
        stats = self._copy_frames(train_real, val_real, train_fake, val_fake)

        # 5. Lưu metadata
        print("\nStep 5: Saving metadata...")
        self._save_metadata(stats, train_real, val_real, train_fake, val_fake)

        # 6. Tóm tắt
        self._print_summary(stats)

        return stats

    def _collect_video_folders(self):
        """Thu thập tất cả video folders"""
        real_path = self.extracted_path / 'real'
        fake_path = self.extracted_path / 'fake'

        real_folders = sorted([f for f in real_path.iterdir() if f.is_dir()])
        fake_folders = sorted([f for f in fake_path.iterdir() if f.is_dir()])

        return real_folders, fake_folders

    def _split_folders(self, folders, train_ratio):
        """Split folders thành train/val"""
        if not folders:
            return [], []

        train_folders, val_folders = train_test_split(
            folders,
            train_size=train_ratio,
            random_state=self.random_seed,
            shuffle=True
        )

        return train_folders, val_folders

    def _create_directory_structure(self):
        """Tạo cấu trúc thư mục train/val"""
        for folder in [self.train_real, self.train_fake, self.val_real, self.val_fake]:
            folder.mkdir(parents=True, exist_ok=True)

        print("  ✓ Directory structure created")

    def _copy_frames(self, train_real, val_real, train_fake, val_fake):
        """Copy frames vào train/val folders"""
        stats = {
            'train_real': 0,
            'train_fake': 0,
            'val_real': 0,
            'val_fake': 0
        }

        # Copy train real
        print("  Copying train real frames...")
        stats['train_real'] = self._copy_folder_frames(
            train_real, self.train_real, "Train Real"
        )

        # Copy train fake
        print("  Copying train fake frames...")
        stats['train_fake'] = self._copy_folder_frames(
            train_fake, self.train_fake, "Train Fake"
        )

        # Copy val real
        print("  Copying val real frames...")
        stats['val_real'] = self._copy_folder_frames(
            val_real, self.val_real, "Val Real"
        )

        # Copy val fake
        print("  Copying val fake frames...")
        stats['val_fake'] = self._copy_folder_frames(
            val_fake, self.val_fake, "Val Fake"
        )

        return stats

    def _copy_folder_frames(self, video_folders, dest_folder, desc):
        """Copy frames từ video folders vào destination folder"""
        frame_count = 0

        for video_folder in tqdm(video_folders, desc=desc, unit="video"):
            # Lấy tất cả frames trong video folder
            frames = list(video_folder.glob("*.jpg"))

            for frame_path in frames:
                # Tạo tên file mới: videoname_framename.jpg
                new_name = f"{video_folder.name}_{frame_path.name}"
                dest_path = dest_folder / new_name

                # Copy file
                shutil.copy2(frame_path, dest_path)
                frame_count += 1

        return frame_count

    def _save_metadata(self, stats, train_real, val_real, train_fake, val_fake):
        """Lưu metadata về dataset"""
        metadata = {
            'random_seed': self.random_seed,
            'train_ratio': self.train_ratio,
            'stats': stats,
            'train_videos': {
                'real': [f.name for f in train_real],
                'fake': [f.name for f in train_fake]
            },
            'val_videos': {
                'real': [f.name for f in val_real],
                'fake': [f.name for f in val_fake]
            }
        }

        # Lưu metadata
        metadata_path = self.output_path / 'dataset_metadata.json'
        with open(metadata_path, 'w') as f:
            json.dump(metadata, f, indent=2)

        print(f"  ✓ Metadata saved to {metadata_path}")

    def _print_summary(self, stats):
        """In tóm tắt dataset"""
        print(f"\n{'='*60}")
        print("DATASET SUMMARY")
        print(f"{'='*60}")

        total_train = stats['train_real'] + stats['train_fake']
        total_val = stats['val_real'] + stats['val_fake']
        total = total_train + total_val

        print(f"\nTRAIN SET:")
        print(f"  Real:  {stats['train_real']:,} frames")
        print(f"  Fake:  {stats['train_fake']:,} frames")
        print(f"  Total: {total_train:,} frames ({total_train/total*100:.1f}%)")

        print(f"\nVAL SET:")
        print(f"  Real:  {stats['val_real']:,} frames")
        print(f"  Fake:  {stats['val_fake']:,} frames")
        print(f"  Total: {total_val:,} frames ({total_val/total*100:.1f}%)")

        print(f"\nOVERALL:")
        print(f"  Total frames: {total:,}")
        print(f"  Real:  {stats['train_real'] + stats['val_real']:,} ({(stats['train_real'] + stats['val_real'])/total*100:.1f}%)")
        print(f"  Fake:  {stats['train_fake'] + stats['val_fake']:,} ({(stats['train_fake'] + stats['val_fake'])/total*100:.1f}%)")

        print(f"\n{'='*60}\n")

# ==================== USAGE EXAMPLE ====================

if __name__ == "__main__":
    # Đường dẫn
    EXTRACTED_PATH = Path("/content/Extracted_Data")
    DATASET_PATH = Path("/content/Processed_Data")
    DATASET_PATH.mkdir(parents=True, exist_ok=True)

    # ==================== PHƯƠNG PHÁP 1: COPY FILES (Khuyến nghị) ====================
    print("Building dataset with file copy...")
    builder = DeepfakeDatasetBuilder(
        extracted_path=EXTRACTED_PATH,
        output_path=DATASET_PATH,
        train_ratio=0.8,      # 80% train, 20% val
        random_seed=42        # Reproducible
    )

    stats = builder.build_dataset()


## 3.3. Dành cho FaceForensics++

In [ ]:
# 1. Cấu hình đường dẫn
from pathlib import Path
FF_ROOT = Path(ff_path) / 'FaceForensics++_23'
DRIVE_SAVE_PATH = Path('/content/Extracted_Data')
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)
print(f"Dữ liệu sẽ được lưu tại thư mục MỚI: {DRIVE_SAVE_PATH}")

In [ ]:
# 2. Định nghĩa hàm cắt khuôn mặt (tương tự FaceForensics++)
import os
import torch
import numpy as np
from facenet_pytorch import MTCNN
from pathlib import Path
from tqdm import tqdm
import cv2

# Import Decord
try:
    from decord import VideoReader, cpu, gpu
    DECORD_AVAILABLE = True
except ImportError:
    print("⚠️  Decord not installed. Install with: pip install decord")
    DECORD_AVAILABLE = False

# ==================== GOOGLE COLAB SINGLE GPU PROCESSOR WITH DECORD ====================

class ColabSingleGPUProcessor:
    """
    Xử lý video với Decord trên Google Colab (Single GPU)
    Tối ưu cho T4/V100/A100 GPU
    """

    def __init__(self, batch_size=32, use_gpu_decode=False):
        """
        Args:
            batch_size: Số frames xử lý cùng lúc trên GPU
            use_gpu_decode: True = GPU decode (cần build Decord), False = CPU decode
        """
        if not DECORD_AVAILABLE:
            raise RuntimeError("Decord is required. Install: pip install decord")

        print("Initializing Google Colab Single GPU Processor...")
        print(f"PyTorch version: {torch.__version__}")
        print(f"CUDA available: {torch.cuda.is_available()}")

        if not torch.cuda.is_available():
            raise RuntimeError("No GPU available! Please enable GPU in Colab: Runtime > Change runtime type > GPU")

        # In thông tin GPU
        props = torch.cuda.get_device_properties(0)
        vram_gb = props.total_memory / 1e9
        print(f"GPU: {props.name} ({vram_gb:.1f}GB VRAM)")

        self.batch_size = batch_size
        self.device = 'cuda:0'

        # Kiểm tra Decord GPU support
        try:
            from decord import bridge
            has_gpu = bridge.bridge_available()
            print(f"\nDecord GPU support: {has_gpu}")
        except:
            has_gpu = False
            print(f"\nDecord GPU support: {has_gpu}")

        # Chọn context cho Decord
        if use_gpu_decode and has_gpu:
            print("✓ Using GPU hardware decoding (NVDEC)")
            self.decord_ctx = gpu(0)
            self.use_gpu_decode = True
        else:
            if use_gpu_decode and not has_gpu:
                print("⚠️  GPU decode requested but not available, falling back to CPU")
            else:
                print("✓ Using CPU decoding")
            self.decord_ctx = cpu(0)
            self.use_gpu_decode = False

        # Khởi tạo MTCNN model
        print("\nInitializing MTCNN model...")
        self.mtcnn = MTCNN(
            image_size=224,
            margin=0,
            min_face_size=20,
            thresholds=[0.6, 0.7, 0.7],
            factor=0.709,
            post_process=True,
            device=self.device,
            keep_all=False
        )
        print("  ✓ MTCNN model ready")

    def process_single_video(self, video_path, output_folder, sample_rate=5):
        """
        Xử lý 1 video với Decord

        Args:
            video_path: Đường dẫn video
            output_folder: Thư mục lưu frames
            sample_rate: Lấy 1 frame mỗi N frames
        """
        output_folder = Path(output_folder)
        output_folder.mkdir(parents=True, exist_ok=True)

        try:
            # Mở video với Decord
            vr = VideoReader(str(video_path), ctx=self.decord_ctx)
            total_frames = len(vr)

            # Tính indices cần lấy
            frame_indices = list(range(0, total_frames, sample_rate))

            if not frame_indices:
                return 0

            saved_count = 0

            # Xử lý theo batch
            for i in range(0, len(frame_indices), self.batch_size):
                batch_idx = frame_indices[i:i + self.batch_size]

                # Load batch frames với Decord (nhanh hơn cv2 nhiều)
                frames_batch = vr.get_batch(batch_idx).asnumpy()

                # Convert thành list
                frame_list = [frames_batch[j] for j in range(len(frames_batch))]

                # Detect faces với MTCNN
                try:
                    with torch.no_grad():
                        boxes, probs = self.mtcnn.detect(frame_list)

                    # Lưu faces
                    for frame, box, prob, idx in zip(frame_list, boxes, probs, batch_idx):
                        if box is not None and prob is not None and prob[0] > 0.9:
                            x1, y1, x2, y2 = [int(b) for b in box[0]]
                            face = frame[y1:y2, x1:x2]

                            if face.size > 0:
                                # Decord trả về RGB, convert sang BGR cho cv2.imwrite
                                face_bgr = cv2.cvtColor(face, cv2.COLOR_RGB2BGR)
                                output_path = output_folder / f"frame_{idx:06d}.jpg"
                                cv2.imwrite(str(output_path), face_bgr)
                                saved_count += 1
                except:
                    pass  # Silent error handling

            return saved_count

        except Exception as e:
            return 0

    def process_videos(self, video_paths, output_base, sample_rate=5):
        """
        Xử lý danh sách videos tuần tự

        Args:
            video_paths: List các đường dẫn video
            output_base: Thư mục output
            sample_rate: Lấy 1 frame mỗi N frames
        """
        output_base = Path(output_base)
        results = {}

        for video_path in tqdm(video_paths, desc="Processing videos", unit="video", leave=True, position=0):
            output_folder = output_base / video_path.stem

            count = self.process_single_video(
                video_path, output_folder, sample_rate
            )

            results[video_path.name] = count


        return results

    def cleanup(self):
        """
        Làm sạch VRAM và giải phóng tài nguyên
        """
        print("\nCleaning up GPU memory...")

        # Xóa cache của PyTorch
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

        # Garbage collection
        import gc
        gc.collect()

        print("✓ GPU memory cleaned")

        print(f"{'='*60}\n")

In [ ]:
# 3. Trích xuất chân dung từ video trong dataset Celeb-DF v2
def process_ff():
    real_out = DRIVE_SAVE_PATH / 'real'
    fake_out = DRIVE_SAVE_PATH / 'fake'
    print("\n=== FF REAL ===")
    real_path = FF_ROOT / "original"
    real_videos = list(real_path.glob("*.mp4"))
    print(f"Found {len(real_videos)} real videos")
    results_real = processor.process_videos(
      real_videos,
      real_out,
      sample_rate=10
    )
    processor.cleanup()

    print("\n=== FF FAKE ===")
    fake_paths = [
        FF_ROOT / "DeepFakeDetection",
        FF_ROOT
        FF_ROOT / "Face2Face",
        FF_ROOT / "FaceSwap",
    ];
    # fake_path = CELEB_ROOT / "Celeb-synthesis"
    for fake_path in fake_paths:
      print(f"\n=== Currently processing: {fake_path.name} ===")
      fake_videos = list(fake_path.glob("*.mp4"))
      print(f"Found {len(fake_videos)} fake videos")
      result_fake = processor.process_videos(
        fake_videos,
        fake_out,
        sample_rate=10
      )
      processor.cleanup()

processor = ColabSingleGPUProcessor(batch_size=32, use_gpu_decode=False)
process_ff()
print(f"Hoàn tất! Kiểm tra thư mục: {DRIVE_SAVE_PATH}")

In [ ]:
import os
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import json
import random

# ==================== DATASET BUILDER ====================

class DeepfakeDatasetBuilder:
    """
    Xây dựng dataset train/val từ thư mục extracted frames
    """

    def __init__(self, extracted_path, output_path, train_ratio=0.8, random_seed=42):
        """
        Args:
            extracted_path: Thư mục chứa real/ và fake/ folders
            output_path: Thư mục output cho dataset
            train_ratio: Tỉ lệ train (0.8 = 80% train, 20% val)
            random_seed: Random seed để reproducible
        """
        self.extracted_path = Path(extracted_path)
        self.output_path = Path(output_path)
        self.train_ratio = train_ratio
        self.random_seed = random_seed

        # Set random seed
        random.seed(random_seed)

        # Tạo cấu trúc thư mục
        self.train_path = self.output_path / 'train'
        self.val_path = self.output_path / 'val'

        self.train_real = self.train_path / 'real'
        self.train_fake = self.train_path / 'fake'
        self.val_real = self.val_path / 'real'
        self.val_fake = self.val_path / 'fake'

        print(f"Dataset Builder initialized:")
        print(f"  Source: {self.extracted_path}")
        print(f"  Output: {self.output_path}")
        print(f"  Train ratio: {train_ratio*100:.0f}%")
        print(f"  Val ratio: {(1-train_ratio)*100:.0f}%")

    def build_dataset(self):
        """
        Xây dựng dataset từ extracted frames
        """
        print(f"\n{'='*60}")
        print("BUILDING DATASET")
        print(f"{'='*60}\n")

        # 1. Thu thập tất cả video folders
        print("Step 1: Collecting video folders...")
        real_folders, fake_folders = self._collect_video_folders()

        print(f"  Found {len(real_folders)} real video folders")
        print(f"  Found {len(fake_folders)} fake video folders")

        # 2. Split train/val theo video-level (không phải frame-level)
        print("\nStep 2: Splitting train/val...")
        train_real, val_real = self._split_folders(real_folders, self.train_ratio)
        train_fake, val_fake = self._split_folders(fake_folders, self.train_ratio)

        print(f"  Train: {len(train_real)} real + {len(train_fake)} fake videos")
        print(f"  Val:   {len(val_real)} real + {len(val_fake)} fake videos")

        # 3. Tạo cấu trúc thư mục
        print("\nStep 3: Creating directory structure...")
        self._create_directory_structure()

        # 4. Copy frames vào train/val folders
        print("\nStep 4: Copying frames...")
        stats = self._copy_frames(train_real, val_real, train_fake, val_fake)

        # 5. Lưu metadata
        print("\nStep 5: Saving metadata...")
        self._save_metadata(stats, train_real, val_real, train_fake, val_fake)

        # 6. Tóm tắt
        self._print_summary(stats)

        return stats

    def _collect_video_folders(self):
        """Thu thập tất cả video folders"""
        real_path = self.extracted_path / 'real'
        fake_path = self.extracted_path / 'fake'

        real_folders = sorted([f for f in real_path.iterdir() if f.is_dir()])
        fake_folders = sorted([f for f in fake_path.iterdir() if f.is_dir()])

        return real_folders, fake_folders

    def _split_folders(self, folders, train_ratio):
        """Split folders thành train/val"""
        if not folders:
            return [], []

        train_folders, val_folders = train_test_split(
            folders,
            train_size=train_ratio,
            random_state=self.random_seed,
            shuffle=True
        )

        return train_folders, val_folders

    def _create_directory_structure(self):
        """Tạo cấu trúc thư mục train/val"""
        for folder in [self.train_real, self.train_fake, self.val_real, self.val_fake]:
            folder.mkdir(parents=True, exist_ok=True)

        print("  ✓ Directory structure created")

    def _copy_frames(self, train_real, val_real, train_fake, val_fake):
        """Copy frames vào train/val folders"""
        stats = {
            'train_real': 0,
            'train_fake': 0,
            'val_real': 0,
            'val_fake': 0
        }

        # Copy train real
        print("  Copying train real frames...")
        stats['train_real'] = self._copy_folder_frames(
            train_real, self.train_real, "Train Real"
        )

        # Copy train fake
        print("  Copying train fake frames...")
        stats['train_fake'] = self._copy_folder_frames(
            train_fake, self.train_fake, "Train Fake"
        )

        # Copy val real
        print("  Copying val real frames...")
        stats['val_real'] = self._copy_folder_frames(
            val_real, self.val_real, "Val Real"
        )

        # Copy val fake
        print("  Copying val fake frames...")
        stats['val_fake'] = self._copy_folder_frames(
            val_fake, self.val_fake, "Val Fake"
        )

        return stats

    def _copy_folder_frames(self, video_folders, dest_folder, desc):
        """Copy frames từ video folders vào destination folder"""
        frame_count = 0

        for video_folder in tqdm(video_folders, desc=desc, unit="video"):
            # Lấy tất cả frames trong video folder
            frames = list(video_folder.glob("*.jpg"))

            for frame_path in frames:
                # Tạo tên file mới: videoname_framename.jpg
                new_name = f"{video_folder.name}_{frame_path.name}"
                dest_path = dest_folder / new_name

                # Copy file
                shutil.copy2(frame_path, dest_path)
                frame_count += 1

        return frame_count

    def _save_metadata(self, stats, train_real, val_real, train_fake, val_fake):
        """Lưu metadata về dataset"""
        metadata = {
            'random_seed': self.random_seed,
            'train_ratio': self.train_ratio,
            'stats': stats,
            'train_videos': {
                'real': [f.name for f in train_real],
                'fake': [f.name for f in train_fake]
            },
            'val_videos': {
                'real': [f.name for f in val_real],
                'fake': [f.name for f in val_fake]
            }
        }

        # Lưu metadata
        metadata_path = self.output_path / 'dataset_metadata.json'
        with open(metadata_path, 'w') as f:
            json.dump(metadata, f, indent=2)

        print(f"  ✓ Metadata saved to {metadata_path}")

    def _print_summary(self, stats):
        """In tóm tắt dataset"""
        print(f"\n{'='*60}")
        print("DATASET SUMMARY")
        print(f"{'='*60}")

        total_train = stats['train_real'] + stats['train_fake']
        total_val = stats['val_real'] + stats['val_fake']
        total = total_train + total_val

        print(f"\nTRAIN SET:")
        print(f"  Real:  {stats['train_real']:,} frames")
        print(f"  Fake:  {stats['train_fake']:,} frames")
        print(f"  Total: {total_train:,} frames ({total_train/total*100:.1f}%)")

        print(f"\nVAL SET:")
        print(f"  Real:  {stats['val_real']:,} frames")
        print(f"  Fake:  {stats['val_fake']:,} frames")
        print(f"  Total: {total_val:,} frames ({total_val/total*100:.1f}%)")

        print(f"\nOVERALL:")
        print(f"  Total frames: {total:,}")
        print(f"  Real:  {stats['train_real'] + stats['val_real']:,} ({(stats['train_real'] + stats['val_real'])/total*100:.1f}%)")
        print(f"  Fake:  {stats['train_fake'] + stats['val_fake']:,} ({(stats['train_fake'] + stats['val_fake'])/total*100:.1f}%)")

        print(f"\n{'='*60}\n")

# ==================== USAGE EXAMPLE ====================

if __name__ == "__main__":
    # Đường dẫn
    EXTRACTED_PATH = Path("/content/Extracted_Data")
    DATASET_PATH = Path("/content/Processed_Data")
    DATASET_PATH.mkdir(parents=True, exist_ok=True)

    # ==================== PHƯƠNG PHÁP 1: COPY FILES (Khuyến nghị) ====================
    print("Building dataset with file copy...")
    builder = DeepfakeDatasetBuilder(
        extracted_path=EXTRACTED_PATH,
        output_path=DATASET_PATH,
        train_ratio=0.8,      # 80% train, 20% val
        random_seed=42        # Reproducible
    )

    stats = builder.build_dataset()


(Tùy chọn) Đoạn code sau cho phép nén thư mục ảnh đã cắt tải về để sử dụng cho lần sau nếu cần thiết.

In [ ]:
import os

# Đường dẫn đến thư mục cần nén
folder_to_compress = '/content/Processed_Data'

# Tên file zip đầu ra
output_zip_file = '/content/Processed_Data.zip'

# Kiểm tra xem thư mục có tồn tại không
if os.path.exists(folder_to_compress):
    print(f"Đang nén thư mục '{folder_to_compress}' thành '{output_zip_file}' với độ nén cao nhất...")
    # Sử dụng lệnh zip với cờ -r (recursive) và -9 (độ nén cao nhất)
    !zip -r -9 {output_zip_file} {folder_to_compress}
    print(f"Hoàn tất nén. File đã được lưu tại: {output_zip_file}")
    # Kiểm tra kích thước file nén
    !du -h {output_zip_file}
else:
    print(f"Lỗi: Thư mục '{folder_to_compress}' không tồn tại. Vui lòng kiểm tra lại đường dẫn.")

# Bước 4: Mô hình Diffusion

## 4.2. Tiền xử lý, huấn luyện (OPTIMIZED FOR GPU)

In [ ]:
from google.colab import drive

# Lệnh này sẽ hiển thị popup yêu cầu quyền truy cập
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch.cuda.amp import autocast, GradScaler

import matplotlib.pyplot as plt

# ============= OPTIMIZED DATASET =============
class DiffConvDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform

        for label_name in ['FAKE', 'REAL']:
            label = 0 if label_name == 'FAKE' else 1
            folder = os.path.join(root_dir, label_name)
            for fname in os.listdir(folder):
                if fname.endswith('.jpg'):
                    self.samples.append((os.path.join(folder, fname), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform:
            img = self.transform(img)

        # ✅ KHÔNG tính spatial difference ở đây
        return img, label


# ✅ SPATIAL DIFFERENCE TRÊN GPU
def compute_spatial_difference_gpu(img_batch):
    """
    Tính spatial difference trực tiếp trên GPU
    Args:
        img_batch: (B, 3, H, W) tensor trên GPU
    Returns:
        diff: (B, 3, H, W) tensor
    """
    B, C, H, W = img_batch.shape

    # Gradient theo x và y
    diff_x = torch.abs(img_batch[:, :, :, 1:] - img_batch[:, :, :, :-1])
    diff_y = torch.abs(img_batch[:, :, 1:, :] - img_batch[:, :, :-1, :])

    # Pad về kích thước gốc
    diff_x = F.pad(diff_x, (0, 1, 0, 0), mode='replicate')
    diff_y = F.pad(diff_y, (0, 0, 0, 1), mode='replicate')

    return diff_x + diff_y


# ============= MODEL COMPONENTS =============
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = x.view(b, c, -1).mean(dim=2)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y


class DiffConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )
        self.se = SEBlock(256)
        self.gap = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        x = self.features(x)
        x = self.se(x)
        x = self.gap(x)
        return x.view(x.size(0), -1)


class VGG16FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(pretrained=True)
        self.features = vgg.features

        for p in self.features.parameters():
            p.requires_grad = False

    def forward(self, x):
        return self.features(x)


class DiffConvNetWithVGG16(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.vgg = VGG16FeatureExtractor()
        self.diff = DiffConvNet()

        # Fine-tune last VGG block
        for p in self.vgg.features[-10:].parameters():
            p.requires_grad = True

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(512 + 256, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, img, diff):
        vgg_feat = self.vgg(img)
        vgg_feat = self.gap(vgg_feat).view(img.size(0), -1)
        diff_feat = self.diff(diff)
        fused = torch.cat([vgg_feat, diff_feat], dim=1)
        return self.classifier(fused)


class FocalLoss(nn.Module):
    def __init__(self, gamma=2):
        super().__init__()
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss()

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        pt = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma) * ce_loss


# ============= TRAINING SETUP =============
IMG_SIZE = 224
BATCH_SIZE = 32  # ✅ Tăng batch size
ACCUMULATION_STEPS = 4  # ✅ Gradient accumulation
NUM_WORKERS = 4  # ✅ Tăng workers

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = DiffConvDataset('/content/drive/MyDrive/Deepfake_Project/Processed_Data_New/train', transform)
val_dataset = DiffConvDataset('/content/drive/MyDrive/Deepfake_Project/Processed_Data_New/val', transform)

# ✅ Tối ưu DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,  # ✅ Giữ workers alive
    prefetch_factor=2  # ✅ Prefetch batches
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE * 2,  # ✅ Batch lớn hơn cho validation
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

# ============= MODEL & OPTIMIZER =============
model = DiffConvNetWithVGG16(num_classes=2).to(DEVICE)
criterion = FocalLoss(gamma=2)

optimizer = torch.optim.AdamW([
    {"params": model.vgg.features[-10:].parameters(), "lr": 1e-5},
    {"params": model.diff.parameters(), "lr": 1e-4},
    {"params": model.classifier.parameters(), "lr": 1e-4}
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# ✅ Mixed Precision Training
scaler = GradScaler()

# ============= TRAINING LOOP =============
EPOCHS = 15
best_acc = 0

os.makedirs('/content/Models', exist_ok=True)

try:
    from tqdm import tqdm
    use_tqdm = True
except ImportError:
    use_tqdm = False
    print("⚠️ tqdm not installed. Progress bar disabled.")

for epoch in range(EPOCHS):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"{'='*50}")

    model.train()
    train_loss = 0.0

    # ✅ Tạo progress bar
    if use_tqdm:
        pbar = tqdm(train_loader, desc=f"Training",
                    bar_format='{l_bar}{bar:30}{r_bar}',
                    ncols=100)
    else:
        pbar = train_loader

    for batch_idx, (img, y) in enumerate(pbar):
        # ✅ Transfer data ngay khi nhận
        img = img.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        # ✅ Tính spatial difference trên GPU
        diff = compute_spatial_difference_gpu(img)

        # ✅ Mixed precision forward pass
        with autocast():
            out = model(img, diff)
            loss = criterion(out, y)
            loss = loss / ACCUMULATION_STEPS  # ✅ Scale loss

        # ✅ Backward với gradient scaling
        scaler.scale(loss).backward()

        # ✅ Gradient accumulation
        if (batch_idx + 1) % ACCUMULATION_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)  # ✅ Hiệu quả hơn

        train_loss += loss.item() * ACCUMULATION_STEPS

        # ✅ Cập nhật progress bar với loss hiện tại
        if use_tqdm:
            current_avg_loss = train_loss / (batch_idx + 1)
            pbar.set_postfix({'loss': f'{current_avg_loss:.4f}'})

    avg_train_loss = train_loss / len(train_loader)
    print(f"📊 Training Loss: {avg_train_loss:.4f}")

    # ============= VALIDATION =============
    model.eval()
    all_preds = []
    all_labels = []

    # ✅ Progress bar cho validation
    if use_tqdm:
        val_pbar = tqdm(val_loader, desc=f"Validation",
                        bar_format='{l_bar}{bar:30}{r_bar}',
                        ncols=100)
    else:
        val_pbar = val_loader

    with torch.no_grad():
        for img, y in val_pbar:
            img = img.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            # ✅ Spatial difference trên GPU
            diff = compute_spatial_difference_gpu(img)

            with autocast():
                out = model(img, diff)

            # ✅ Accumulate trên GPU
            _, preds = torch.max(out, 1)
            all_preds.append(preds)
            all_labels.append(y)

    # ✅ Transfer 1 lần cuối cùng
    all_preds = torch.cat(all_preds).cpu().numpy()
    all_labels = torch.cat(all_labels).cpu().numpy()

    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='binary')
    recall = recall_score(all_labels, all_preds, average='binary')
    f1 = f1_score(all_labels, all_preds, average='binary')

    print(f"\n📊 Validation Metrics:")
    print(f"   Accuracy:  {acc:.4f}")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall:    {recall:.4f}")
    print(f"   F1-Score:  {f1:.4f}")

    scheduler.step()

    # ✅ Save best model
    if acc > best_acc:
        best_acc = acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_acc': best_acc,
        }, '/content/Models/diff_vgg_best.pth')
        print(f"✅ Saved best model with accuracy: {best_acc:.4f}")

print(f"\n{'='*50}")
print(f"Training completed! Best Accuracy: {best_acc:.4f}")
print(f"{'='*50}")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_We


Epoch 1/15


Training:   0%|                              | 0/290 [00:00<?, ?it/s]/tmp/ipython-input-1256346154.py:263: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Training: 100%|██████████████████████████████| 290/290 [02:36<00:00,  1.85it/s, loss=0.0795]


📊 Training Loss: 0.0795


...


📊 Validation Metrics:
   Accuracy:  0.8492
   Precision: 0.6392
   Recall:    0.5000
   F1-Score:  0.5611

Epoch 15/15


Training:   0%|                              | 0/290 [00:00<?, ?it/s]/tmp/ipython-input-1256346154.py:263: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Training: 100%|██████████████████████████████| 290/290 [01:36<00:00,  3.02it/s, loss=0.0065]


📊 Training Loss: 0.0065


Validation:   0%|                              | 0/37 [00:00<?, ?it/s]/tmp/ipython-input-1256346154.py:308: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Validation: 100%|██████████████████████████████| 37/37 [00:22<00:00,  1.68it/s]



📊 Validation Metrics:
   Accuracy:  0.8552
   Precision: 0.6772
   Recall:    0.4756
   F1-Score:  0.5587
✅ Saved best model with accuracy: 0.8552

Training completed! Best Accuracy: 0.8552


## 4.3. Đánh giá mô hình

In [ ]:
model.load_state_dict(torch.load('/content/Models/diff_vgg_best.pth'))
model.eval()

preds, gts = [], []

with torch.no_grad():
    for img, diff, y in val_loader:
        out = model(img, diff)
        _, p = torch.max(out, 1)
        preds.extend(p.cpu().numpy())
        gts.extend(y.numpy())

acc  = accuracy_score(gts, preds)
prec = precision_score(gts, preds, pos_label=0)
rec  = recall_score(gts, preds, pos_label=0)
f1   = f1_score(gts, preds, pos_label=0)

print("\n==============================")
print("📊 KẾT QUẢ DIFF + VGG16")
print("==============================")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")

cm = confusion_matrix(gts, preds)
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=['FAKE', 'REAL'],
            yticklabels=['FAKE', 'REAL'],
            cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix – Diff + VGG16")
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: '/content/Models/diff_vgg_best.pth'

# BASELINE 1A: Xception (nguyên bản)

## Bước B1A.1: Tiền xử lý, huấn luyện
Code này sẽ load ảnh từ Drive, tiền xử lý (chuẩn bị dữ liệu) train Xception và lưu model tốt nhất vào Drive.

In [ ]:
# --- CODE PHẦN 2: TRAINING XCEPTION Nguyên Bản ---
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import timm
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
from tqdm import tqdm

# 1. Cấu hình
BATCH_SIZE = 32
EPOCHS = 3 # Tăng lên nếu muốn kết quả tốt hơn
LR = 0.001
IMG_SIZE = 299 # Xception chuẩn thường dùng 299x299
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
################ THAY ĐỔI ĐƯỜNG DẪN DATASET Ở ĐÂY ###################
DATA_PATH = '/content/Processed_Data'
#####################################################################
MODEL_SAVE_PATH = '/content/Models/xception_best.pth'
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

# 2. Chuẩn bị dữ liệu (Data Augmentation & Loader)
data_transforms = {
    'train': transforms.Compose([

        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ]),
    'val': transforms.Compose([

        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ]),
}

image_datasets = {x: datasets.ImageFolder(os.path.join(DATA_PATH, x), data_transforms[x]) for x in ['train', 'val']}
dataloaders = {x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE, shuffle=(x=='train'), num_workers=2) for x in ['train', 'val']}
class_names = image_datasets['train'].classes # ['FAKE', 'REAL'] -> Lưu ý thứ tự này

# 3. Khởi tạo mô hình Xception (dùng thư viện timm)
print("Đang tải mô hình Xception...")
model = timm.create_model('xception', pretrained=True, num_classes=2)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# 4. Hàm đánh giá chỉ số
def calculate_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='binary', pos_label=0) # 0 là FAKE (dựa vào thứ tự folder alphabet) hoặc check class_names
    rec = recall_score(y_true, y_pred, average='binary', pos_label=0)
    f1 = f1_score(y_true, y_pred, average='binary', pos_label=0)
    return acc, prec, rec, f1

# Lưu ý: Folder thường xếp Alphabet: FAKE, REAL.
# dataset.class_to_idx: {'FAKE': 0, 'REAL': 1}
# Vậy pos_label=0 nghĩa là ta đang coi FAKE là Positive class (lớp cần phát hiện).

# 5. Vòng lặp Training
best_acc = 0.0

for epoch in range(EPOCHS):
    print(f'Epoch {epoch+1}/{EPOCHS}')
    print('-' * 10)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        all_preds = []
        all_labels = []

        for inputs, labels in tqdm(dataloaders[phase]):
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        epoch_loss = running_loss / len(image_datasets[phase])
        acc, prec, rec, f1 = calculate_metrics(all_labels, all_preds)

        print(f'{phase} Loss: {epoch_loss:.4f} Acc: {acc:.4f} Prec: {prec:.4f} Rec: {rec:.4f} F1: {f1:.4f}')

        if phase == 'val' and acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f"--> Đã lưu model tốt nhất với Acc: {best_acc:.4f}")

print("Hoàn tất Training Xception!")

## Bước B1A.2: Đánh giá mô hình Xception nguyên bản đã train trên tập ảnh

In [ ]:
import torch
import timm
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# 1. Cấu hình đường dẫn và tham số XCEPTION
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_PATH = '/content/Models/xception_best.pth'
DATA_PATH = '/content/Processed_Data/val'

# --- QUAN TRỌNG: Tham số chuẩn của Xception ---
IMG_SIZE = 299
NORM_MEAN = [0.5, 0.5, 0.5]
NORM_STD = [0.5, 0.5, 0.5]

print("🚀 Đang khởi tạo đánh giá XCEPTION...")

# 2. Chuẩn bị dữ liệu
test_transform = transforms.Compose([

    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD)
])

try:
    test_dataset = datasets.ImageFolder(DATA_PATH, test_transform)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
    class_names = test_dataset.classes
except Exception as e:
    print(f"❌ Lỗi đọc dữ liệu: {e}")
    exit()

# 3. Load Model
try:
    model = timm.create_model('xception', pretrained=False, num_classes=2)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    print("✅ Đã load weights Xception thành công!")
except Exception as e:
    print(f"❌ Lỗi load model: {e}")
    exit()

# 4. Dự đoán
all_preds = []
all_labels = []

print("⏳ Đang chạy dự đoán trên tập Test...")
with torch.no_grad():
    for inputs, labels in tqdm(test_loader):
        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# 5. Tính chỉ số (Ưu tiên nhãn FAKE)
fake_idx = test_dataset.class_to_idx['FAKE']
acc = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, pos_label=fake_idx)
rec = recall_score(all_labels, all_preds, pos_label=fake_idx)
f1 = f1_score(all_labels, all_preds, pos_label=fake_idx)

print("\n" + "="*30)
print("📊 KẾT QUẢ XCEPTION")
print("="*30)
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")

# Vẽ Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Xception')
plt.ylabel('Thực tế')
plt.xlabel('Dự đoán')
plt.show()

# [baseline 1B]: Xception lai (kết hợp với features từ VGG16)

## Bước B1B.1: Tiền xử lý, huấn luyện
Code này sẽ load ảnh từ Drive, tiền xử lý (chuẩn bị dữ liệu) train Xception và lưu model tốt nhất vào Drive.

In [ ]:
# --- CODE PHẦN 2: TRAINING XCEPTION Lai ---
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import timm
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
from tqdm import tqdm


In [ ]:
# 1. Cấu hình
BATCH_SIZE = 32
LR = 0.001
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
########################## THAY ĐỔI ĐƯỜNG DẪN DATASET Ở ĐÂY ##################################
DATA_PATH = '/content/Processed_Data'
##############################################################################################
MODEL_SAVE_PATH = '/content/Models/xception_custom_best.pth'
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)


In [ ]:

# 2. Chuẩn bị dữ liệu (Data Augmentation & Loader)
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ]),
    'val': transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ]),
}

image_datasets = {x: datasets.ImageFolder(os.path.join(DATA_PATH, x), data_transforms[x]) for x in ['train', 'val']}
dataloaders = {x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE, shuffle=(x=='train'), num_workers=2) for x in ['train', 'val']}
class_names = image_datasets['train'].classes # ['FAKE', 'REAL'] -> Lưu ý thứ tự này


In [ ]:
# 3. Định nghĩa lớp VGG16FeatureExtractor
class VGG16FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(pretrained=True)

        # Chỉ lấy phần convolutional
        self.features = vgg.features

        # Freeze toàn bộ VGG16
        for param in self.features.parameters():
            param.requires_grad = False

    def forward(self, x):
        """ Example:
        x: (B, 3, 224, 224)
        return: (B, 512, 7, 7)
        """
        return self.features(x)

In [ ]:
# 4. Định nghĩa mô hình Xception tùy chỉnh với VGG16 Feature Extractor
class CustomXceptionWithVGG16(nn.Module):
    def __init__(self, num_classes=2):
        super(CustomXceptionWithVGG16, self).__init__()

        # Khởi tạo VGG16 Feature Extractor
        self.vgg16_features = VGG16FeatureExtractor()

        # Lấy Xception model từ timm (không bao gồm classifier)
        # FIX: Đăng ký xception_backbone như một submodule để nó được chuyển lên GPU
        self.xception_core = timm.create_model('xception', pretrained=True, num_classes=0)  # Make it a submodule

        # Adaptive pooling để chuyển VGG16 features (B, 512, 7, 7) thành vector
        self.vgg_adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.vgg_flatten = nn.Flatten()

        # Lấy số chiều output của Xception features
        # Tạo một dummy input để kiểm tra kích thước
        with torch.no_grad():
            dummy_input = torch.randn(1, 3, 299, 299)
            # Sử dụng phương thức forward_features của submodule đã đăng ký
            xception_feat = self.xception_core.forward_features(dummy_input) if hasattr(self.xception_core, 'forward_features') else self.xception_core(dummy_input)
            if isinstance(xception_feat, tuple):
                xception_feat = xception_feat[0]
            xception_dim = xception_feat.shape[1] if len(xception_feat.shape) > 1 else xception_feat.numel()

        # VGG16 features sau adaptive pooling: 512
        vgg_dim = 512

        # Kết hợp features từ VGG16 và Xception
        combined_dim = vgg_dim + xception_dim

        # Classifier cuối cùng
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(combined_dim, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        # VGG16 Feature Extractor (x: B, 3, 299, 299)
        # Cần resize về 224x224 cho VGG16 hoặc dùng interpolation
        x_vgg = nn.functional.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)
        vgg_features = self.vgg16_features(x_vgg)  # (B, 512, 7, 7)
        vgg_features = self.vgg_adaptive_pool(vgg_features)  # (B, 512, 1, 1)
        vgg_features = self.vgg_flatten(vgg_features)  # (B, 512)

        # Xception features (x: B, 3, 299, 299)
        # Gọi phương thức forward_features trên submodule đã đăng ký
        xception_features = self.xception_core.forward_features(x) if hasattr(self.xception_core, 'forward_features') else self.xception_core(x)
        if isinstance(xception_features, tuple):
            xception_features = xception_features[0]

        # Global average pooling cho Xception features nếu cần
        if len(xception_features.shape) > 2:
            xception_features = nn.functional.adaptive_avg_pool2d(xception_features, (1, 1))
            xception_features = xception_features.view(xception_features.size(0), -1)

        # Kết hợp features
        combined_features = torch.cat([vgg_features, xception_features], dim=1)

        # Classifier
        output = self.classifier(combined_features)

        return output

# Khởi tạo mô hình tùy chỉnh
print("Đang tải mô hình Xception tùy chỉnh với VGG16 Feature Extractor...")
model = CustomXceptionWithVGG16(num_classes=2)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)


In [ ]:
# 5. Hàm đánh giá chỉ số
def calculate_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='binary', pos_label=0) # 0 là FAKE (dựa vào thứ tự folder alphabet) hoặc check class_names
    rec = recall_score(y_true, y_pred, average='binary', pos_label=0)
    f1 = f1_score(y_true, y_pred, average='binary', pos_label=0)
    return acc, prec, rec, f1

# Lưu ý: Folder thường xếp Alphabet: FAKE, REAL.
# dataset.class_to_idx: {'FAKE': 0, 'REAL': 1}
# Vậy pos_label=0 nghĩa là ta đang coi FAKE là Positive class (lớp cần phát hiện).


In [ ]:
EPOCHS = 3 # Tăng lên nếu muốn kết quả tốt hơn
# 6. Vòng lặp Training
best_acc = 0.0

for epoch in range(EPOCHS):
    print(f'Epoch {epoch+1}/{EPOCHS}')
    print('-' * 10)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        all_preds = []
        all_labels = []

        for inputs, labels in tqdm(dataloaders[phase]):
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        epoch_loss = running_loss / len(image_datasets[phase])
        acc, prec, rec, f1 = calculate_metrics(all_labels, all_preds)

        print(f'{phase} Loss: {epoch_loss:.4f} Acc: {acc:.4f} Prec: {prec:.4f} Rec: {rec:.4f} F1: {f1:.4f}')

        if phase == 'val' and acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f"--> Đã lưu model tốt nhất với Acc: {best_acc:.4f}")

print("Hoàn tất Training Xception!")

## Bước B1B.2: Đánh giá mô hình Xception lai đã train trên tập ảnh  
(điều kiện: model được lưu tại `/content/Models/xception_best.pth`)

In [ ]:
import torch
import timm
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# 1. Cấu hình đường dẫn và tham số XCEPTION
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_PATH = '/content/Models/xception_custom_best.pth'
DATA_PATH = '/content/Processed_Data/val'

# --- QUAN TRỌNG: Tham số chuẩn của Xception ---
# CustomXceptionWithVGG16 cũng sử dụng IMG_SIZE 299 cho phần Xception
IMG_SIZE = 299
NORM_MEAN = [0.5, 0.5, 0.5]
NORM_STD = [0.5, 0.5, 0.5]

print("🚀 Đang khởi tạo đánh giá XCEPTION...")

# 2. Chuẩn bị dữ liệu
test_transform = transforms.Compose([

    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD)
])

try:
    test_dataset = datasets.ImageFolder(DATA_PATH, test_transform)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
    class_names = test_dataset.classes
except Exception as e:
    print(f"❌ Lỗi đọc dữ liệu: {e}")
    exit()

# 3. Load Model
try:
    model = CustomXceptionWithVGG16(num_classes=2)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    print("✅ Đã load weights Xception thành công!")
except Exception as e:
    print(f"❌ Lỗi load model: {e}")
    exit()

# 4. Dự đoán
all_preds = []
all_labels = []

print("⏳ Đang chạy dự đoán trên tập Test...")
with torch.no_grad():
    for inputs, labels in tqdm(test_loader):
        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# 5. Tính chỉ số (Ưu tiên nhãn FAKE)
fake_idx = test_dataset.class_to_idx['FAKE']
acc = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, pos_label=fake_idx)
rec = recall_score(all_labels, all_preds, pos_label=fake_idx)
f1 = f1_score(all_labels, all_preds, pos_label=fake_idx)

print("\n" + "="*30)
print("📊 KẾT QUẢ XCEPTION")
print("="*30)
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")

# Vẽ Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Xception')
plt.ylabel('Thực tế')
plt.xlabel('Dự đoán')
plt.show()

# BASELINE 2A: EfficientNet (nguyên bản)
Phần này tương tự Xception, nhưng thay đổi kiến trúc mạng sang EfficientNet (phiên bản B0 cho nhẹ).

## Bước B2A.1: Tiền xử lý và huấn luyện

In [ ]:
# --- CODE PHẦN 3: TRAINING EFFICIENTNET ---
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import timm
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
from tqdm import tqdm

# 1. Cấu hình
BATCH_SIZE = 32
EPOCHS = 3
LR = 0.001
IMG_SIZE = 224 # EfficientNet B0 thường dùng 224x224
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_PATH = '/content/Processed_Data' #THAY ĐỔI ĐƯỜNG DẪN DATASET Ở ĐÂY
MODEL_SAVE_PATH = '/content/Models/efficientnet_best.pth'

# 2. Chuẩn bị dữ liệu (Lưu ý kích thước ảnh khác Xception)
data_transforms = {
    'train': transforms.Compose([

        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # Chuẩn ImageNet
    ]),
    'val': transforms.Compose([

        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

image_datasets = {x: datasets.ImageFolder(os.path.join(DATA_PATH, x), data_transforms[x]) for x in ['train', 'val']}
dataloaders = {x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE, shuffle=(x=='train'), num_workers=2) for x in ['train', 'val']}

# 3. Khởi tạo mô hình EfficientNet-B0
print("Đang tải mô hình EfficientNet-B0...")
# Sử dụng 'tf_efficientnet_b0_ns' (Noisy Student) cho hiệu năng tốt
model = timm.create_model('tf_efficientnet_b0_ns', pretrained=True, num_classes=2)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# 4. Hàm đánh giá (Giống phần Xception)
def calculate_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    # Giả sử thư mục FAKE xếp trước REAL -> index 0 là FAKE
    prec = precision_score(y_true, y_pred, average='binary', pos_label=0)
    rec = recall_score(y_true, y_pred, average='binary', pos_label=0)
    f1 = f1_score(y_true, y_pred, average='binary', pos_label=0)
    return acc, prec, rec, f1

# 5. Vòng lặp Training
best_acc = 0.0

for epoch in range(EPOCHS):
    print(f'Epoch {epoch+1}/{EPOCHS}')
    print('-' * 10)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        all_preds = []
        all_labels = []

        for inputs, labels in tqdm(dataloaders[phase]):
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        epoch_loss = running_loss / len(image_datasets[phase])
        acc, prec, rec, f1 = calculate_metrics(all_labels, all_preds)

        print(f'{phase} Loss: {epoch_loss:.4f} Acc: {acc:.4f} Prec: {prec:.4f} Rec: {rec:.4f} F1: {f1:.4f}')

        if phase == 'val' and acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f"--> Đã lưu model EfficientNet tốt nhất với Acc: {best_acc:.4f}")

print("Hoàn tất Training EfficientNet!")

## Bước B2A.2: Đánh giá mô hình EfficientNetB0 đã train trên tập ảnh

In [ ]:
import torch
import timm
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# 1. Cấu hình đường dẫn và tham số EFFICIENTNET
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_PATH = '/content/Models/efficientnet_best.pth'
DATA_PATH = '/content/Processed_Data/val'

# --- QUAN TRỌNG: Tham số chuẩn của EfficientNet ---
IMG_SIZE = 224
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

print("🚀 Đang khởi tạo đánh giá EFFICIENTNET...")

# 2. Chuẩn bị dữ liệu
test_transform = transforms.Compose([

    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD)
])

try:
    test_dataset = datasets.ImageFolder(DATA_PATH, test_transform)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
    class_names = test_dataset.classes
except Exception as e:
    print(f"❌ Lỗi đọc dữ liệu: {e}")
    exit()

# 3. Load Model
try:
    # Lưu ý tên model phải khớp với lúc train (tf_efficientnet_b0_ns)
    model = timm.create_model('tf_efficientnet_b0_ns', pretrained=False, num_classes=2)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    print("✅ Đã load weights EfficientNet thành công!")
except Exception as e:
    print(f"❌ Lỗi load model: {e}")
    exit()

# 4. Dự đoán
all_preds = []
all_labels = []

print("⏳ Đang chạy dự đoán trên tập Test...")
with torch.no_grad():
    for inputs, labels in tqdm(test_loader):
        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# 5. Tính chỉ số
fake_idx = test_dataset.class_to_idx['FAKE']
acc = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, pos_label=fake_idx)
rec = recall_score(all_labels, all_preds, pos_label=fake_idx)
f1 = f1_score(all_labels, all_preds, pos_label=fake_idx)

print("\n" + "="*30)
print("📊 KẾT QUẢ EFFICIENTNET")
print("="*30)
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")

# Vẽ Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - EfficientNet')
plt.ylabel('Thực tế')
plt.xlabel('Dự đoán')
plt.show()

# [baseline 2B]: EfficientNet lai (kết hợp với features từ VGG16)

## Bước B2B.1: Tiền xử lý và huấn luyện

In [ ]:
# --- CODE PHẦN 3: TRAINING EFFICIENTNET ---
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import timm
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
from tqdm import tqdm


In [ ]:
!mkdir /content/Processed_Data
!mkdir /content/Models/

In [ ]:

# 1. Cấu hình
BATCH_SIZE = 32
EPOCHS = 3
LR = 0.001
IMG_SIZE = 224 # EfficientNet B0 thường dùng 224x224
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_PATH = '/content/Processed_Data' #THAY ĐỔI ĐƯỜNG DẪN DATASET Ở ĐÂY
MODEL_SAVE_PATH = '/content/Models/efficientnet_custom_best.pth'


In [ ]:

# 2. Chuẩn bị dữ liệu (Lưu ý kích thước ảnh khác Xception)
data_transforms = {
    'train': transforms.Compose([

        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # Chuẩn ImageNet
    ]),
    'val': transforms.Compose([

        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

image_datasets = {x: datasets.ImageFolder(os.path.join(DATA_PATH, x), data_transforms[x]) for x in ['train', 'val']}
dataloaders = {x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE, shuffle=(x=='train'), num_workers=2) for x in ['train', 'val']}


In [ ]:
# 3. Định nghĩa lớp VGG16FeatureExtractor
class VGG16FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(pretrained=True)

        # Chỉ lấy phần convolutional
        self.features = vgg.features

        # Freeze toàn bộ VGG16
        for param in self.features.parameters():
            param.requires_grad = False

    def forward(self, x):
        """ Example:
        x: (B, 3, 224, 224)
        return: (B, 512, 7, 7)
        """
        return self.features(x)

In [ ]:
import torch
import torch.nn as nn
import timm
from torchvision import models

# 4. Định nghĩa mô hình Efficientnet_b0 tùy hỉnh với VGG16 Feature Extractor
class CustomEfficientnet_b0_WithVGG16(nn.Module):
    def __init__(self, num_classes=2, device=None):
        super().__init__()
        # Xác định thiết bị (nếu không truyền vào thì mặc định dùng CUDA nếu có)
        if device is None:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.device = device

        # VGG16 Feature Extractor trên device
        vgg = models.vgg16(pretrained=True)
        self.vgg_features = vgg.features.to(self.device)
        for param in self.vgg_features.parameters():
            param.requires_grad = False

        # EfficientNet-B0 Feature Extractor trên device
        self.effnet = timm.create_model('tf_efficientnet_b0_ns', pretrained=True, num_classes=0)  # no head
        self.effnet = self.effnet.to(self.device)

        # Head: tùy chỉnh Linear nhận đầu vào là tổng số features
        vgg_feat_dim = 512 * 7 * 7 # VGG16 output (B, 512, 7, 7) flattened
        effnet_feat_dim = 1280  # EfficientNet-B0 last feature dim (after pooling and flattening)

        self.classifier = nn.Sequential(
            nn.Linear(vgg_feat_dim + effnet_feat_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        ).to(self.device)

    def forward(self, x):
        # Đảm bảo input cũng trên device
        x = x.to(self.device)

        # VGG16 features
        # VGG16 expects 224x224. Since input is IMG_SIZE (224), no explicit resize here.
        vgg_feat = self.vgg_features(x)           # (B, 512, 7, 7)
        vgg_feat_flat = vgg_feat.view(vgg_feat.size(0), -1) # Flatten to (B, 512*7*7)

        # EfficientNet features
        eff_feat = self.effnet.forward_features(x)  # (B, 1280, 7, 7) or similar if AdaptiveAvgPool2d not applied
        # Apply Global Average Pooling and flatten to ensure 2D tensor (Batch, Features)
        eff_feat_flat = nn.functional.adaptive_avg_pool2d(eff_feat, (1, 1))
        eff_feat_flat = eff_feat_flat.view(eff_feat_flat.size(0), -1) # Flatten to (B, 1280)

        # Concatenate
        concat_feat = torch.cat([vgg_feat_flat, eff_feat_flat], dim=1)

        # Classifier head
        out = self.classifier(concat_feat)
        return out

# 4. Khởi tạo mô hình Efficientnet_b0 tùy chỉnh với VGG16 Feature Extractor
print("Đang tải mô hình Efficientnet_b0 tùy chỉnh với VGG16 Feature Extractor...")
model = CustomEfficientnet_b0_WithVGG16(num_classes=2)
model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)


In [ ]:
# 5. Hàm đánh giá (Giống phần Xception)
def calculate_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    # Giả sử thư mục FAKE xếp trước REAL -> index 0 là FAKE
    prec = precision_score(y_true, y_pred, average='binary', pos_label=0)
    rec = recall_score(y_true, y_pred, average='binary', pos_label=0)
    f1 = f1_score(y_true, y_pred, average='binary', pos_label=0)
    return acc, prec, rec, f1

# 6. Vòng lặp Training
best_acc = 0.0

for epoch in range(EPOCHS):
    print(f'Epoch {epoch+1}/{EPOCHS}')
    print('-' * 10)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        all_preds = []
        all_labels = []

        for inputs, labels in tqdm(dataloaders[phase]):
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        epoch_loss = running_loss / len(image_datasets[phase])
        acc, prec, rec, f1 = calculate_metrics(all_labels, all_preds)

        print(f'{phase} Loss: {epoch_loss:.4f} Acc: {acc:.4f} Prec: {prec:.4f} Rec: {rec:.4f} F1: {f1:.4f}')

        if phase == 'val' and acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f"--> Đã lưu model EfficientNet tốt nhất với Acc: {best_acc:.4f}")

print("Hoàn tất Training EfficientNet!")

## Bước B2B.2: Đánh giá mô hình EfficientNetB0 lai VGG16 đã train trên tập ảnh

In [ ]:
import torch
import timm
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# 1. Cấu hình đường dẫn và tham số EFFICIENTNET
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_PATH = '/content/Models/efficientnet_custom_best.pth'
DATA_PATH = '/content/Processed_Data/val'

# --- QUAN TRỌNG: Tham số chuẩn của EfficientNet ---
IMG_SIZE = 224
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

print("🚀 Đang khởi tạo đánh giá EFFICIENTNET...")

# 2. Chuẩn bị dữ liệu
test_transform = transforms.Compose([

    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD)
])

try:
    test_dataset = datasets.ImageFolder(DATA_PATH, test_transform)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
    class_names = test_dataset.classes
except Exception as e:
    print(f"❌ Lỗi đọc dữ liệu: {e}")
    exit()

# 3. Load Model
try:
    # Lưu ý tên model phải khớp với lúc train
    model = CustomEfficientnet_b0_WithVGG16(num_classes=2)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    print("✅ Đã load weights EfficientNet thành công!")
except Exception as e:
    print(f"❌ Lỗi load model: {e}")
    exit()

# 4. Dự đoán
all_preds = []
all_labels = []

print("⏳ Đang chạy dự đoán trên tập Test...")
with torch.no_grad():
    for inputs, labels in tqdm(test_loader):
        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# 5. Tính chỉ số
fake_idx = test_dataset.class_to_idx['FAKE']
acc = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, pos_label=fake_idx)
rec = recall_score(all_labels, all_preds, pos_label=fake_idx)
f1 = f1_score(all_labels, all_preds, pos_label=fake_idx)

print("\n" + "="*30)
print("📊 KẾT QUẢ EFFICIENTNET")
print("="*30)
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")

# Vẽ Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - EfficientNet')
plt.ylabel('Thực tế')
plt.xlabel('Dự đoán')
plt.show()

# Bước 5. Minh họa case-study bằng LIME

## 5.1. Tải lên thủ công 1 video muốn dùng làm case-study

In [ ]:
from google.colab import files
files.upload()

In [ ]:
# Tìm file mới nhất được tải lên
import glob
import os
import shutil

video_files = glob.glob("*.mp4")

if video_files:
    video_files.sort(key=os.path.getmtime, reverse=True)
    latest_file = video_files[0]

    dest_path = "/content/deepfake.mp4"
    shutil.copy(latest_file, dest_path)
    print(f"✅ Đã xử lý xong file mới nhất: {latest_file}")
else:
    print("❌ Không tìm thấy file nào để copy.")

## 5.2. Cài đặt và tiền xử lý riêng cho case-study

In [ ]:
# Cài đặt các thư viện hỗ trợ đọc video codec lạ và nhận diện khuôn mặt
!pip install -q imageio imageio-ffmpeg

In [ ]:
!mkdir -p /content/CaseStudy_Faces

In [ ]:
import os
import cv2
import torch
import shutil
from PIL import Image
from facenet_pytorch import MTCNN
from google.colab import drive, files

# 1. MOUNT DRIVE VÀ SETUP (Chỉ chạy 1 lần)
# if not os.path.exists('/content/drive'):
#     drive.mount('/content/drive')

# 2. KHỞI TẠO AI NHẬN DIỆN (Bắt buộc phải có dòng này code mới chạy)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

########### Chỉnh lại Image_size = 299 (Xception), 224 (EfficientNetB0, Diffusion) TRƯỚC KHI CHẠY ##############
mtcnn = MTCNN(image_size=299, margin=20, device=device)
################################################################################################################

# 3. ĐỊNH NGHĨA HÀM TRÍCH XUẤT (Logic lấy theo tỷ lệ video)
def extract_multiple_faces(video_path, save_dir, frames_per_video=30):
    if not os.path.exists(video_path):
        print(f"❌ LỖI: Không tìm thấy file video tại {video_path}")
        return 0

    os.makedirs(save_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        print("❌ LỖI: Video không có dữ liệu hoặc định dạng không hỗ trợ.")
        return 0

    # Tính khoảng cách giữa các khung hình để quét hết video
    step = max(1, total_frames // frames_per_video)

    count = 0
    print(f"⏳ Đang xử lý video: {os.path.basename(video_path)}")
    print(f"📊 Tổng số khung hình: {total_frames}. Bước nhảy là {step} khung hình.")

    for i in range(0, total_frames, step):
        if count >= frames_per_video: break

        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if not ret: break

        # Chuyển đổi định dạng cho AI
        img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        save_path = os.path.join(save_dir, f"face_{count}.jpg")

        # AI quét tìm mặt
        face = mtcnn(img, save_path=save_path)

        if face is not None:
            count += 1
            if count % 5 == 0: print(f"✅ Đã trích xuất được {count} khung hình...")

    cap.release()
    print(f"✨ HOÀN TẤT: Đã lưu {count} khung hình vào {save_dir}")
    return count

# 4. THỰC THI (Khớp với file bạn upload)
# Đường dẫn file trên Drive sau khi bạn dùng lệnh shutil.copy ở trên
VIDEO_INPUT = "/content/deepfake.mp4"
FACE_OUTPUT = "/content/CaseStudy_Faces"

num_faces = extract_multiple_faces(VIDEO_INPUT, FACE_OUTPUT, frames_per_video=30)

## 5.3. Khởi tạo và load model đã train

In [ ]:

import torch
import timm
import os
from torchvision import transforms
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

####### !!! THAY ĐƯỜNG DẪN ĐẾN MÔ HÌNH MONG MUỐN TRƯỚC KHI CHẠY !!! ########
## Diffusion: '/content/Models/diffconvnet_best.pth'
## EfficientNet (nguyên bản): '/content/Models/efficientnet_best.pth'
## EfficientNet (với VGG16): '/content/Models/efficientnet_custom_best.pth'
## Xception (nguyên bản): '/content/Models/xception_best.pth'
## Xception (với VGG16): '/content/Models/xception_custom_best.pth'
############################################################################
MODEL_PATH = '/content/Models/xception_best.pth'


###### !!! KHỞI TẠO ĐÚNG MÔ HÌNH TƯƠNG ỨNG TRƯỚC KHI CHẠY !!! ################
# Diffusion (nguyên bản): case_model = DiffConvNet().to(DEVICE)
# Diffusion (với VGG16): case_model = DiffConvNetWithVGG16(num_classes=2, device=DEVICE).to(DEVICE)
# Xception (nguyên bản): case_model = timm.create_model('xception', pretrained=False, num_classes=2)
# Xception (với VGG16): case_model = CustomXceptionWithVGG16(num_classes=2)
# EfficientNet (nguyên bản): case_model = timm.create_model('tf_efficientnet_b0_ns', pretrained=False, num_classes=2)
# EfficientNet (với VGG16): case_model = CustomEfficientnet_b0_WithVGG16(num_classes=2)
case_model = CustomXceptionWithVGG16(num_classes=2)

if os.path.exists(MODEL_PATH):
    # Nạp weights từ bộ nhớ cục bộ /content/ thay vì Drive
    case_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    case_model.to(DEVICE)
    case_model.eval()
    print("✅ Đã kết nối thành công Model từ bộ nhớ Colab!")
else:
    print(f"❌ Không tìm thấy model tại {MODEL_PATH}. Hãy kiểm tra lại Bước 3 đã chạy xong chưa.")

## 5.4. Chạy dự đoán cho case-study

In [ ]:
import numpy as np
import torch.nn.functional as F

# 1. Định nghĩa bộ tiền xử lý ảnh (Giả định bạn đang dùng EfficientNet)
### Nếu dùng Xception, hãy đổi Normalize thành [0.5, 0.5, 0.5],[0.5, 0.5, 0.5]
### Nếu dùng EfficientNet, hãy đổi Normalize thành [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
### Nếu dùng Diffusion, hãy comment đoạn code này đi
case_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)), # Thêm dòng này để resize ảnh về 224x224
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

#### Nếu dùng Diffusion, hãy uncomment đoạn dưới đây sau khi comment đoạn liền trước
# case_transform = transforms.Compose([
#     transforms.ToPILImage(),
#     transforms.ToTensor(),
# ])

# 2. Xác định Index lớp FAKE (Mặc định là 0)
fake_idx = 0

# 3. Phân tích chi tiết từng khuôn mặt
face_folder = '/content/CaseStudy_Faces'
face_files = sorted([f for f in os.listdir(face_folder) if f.endswith(('.jpg', '.png'))])

list_fake_probs = [] # Danh sách lưu % fake của từng khung hình

if not face_files:
    print("❌ Không tìm thấy ảnh trong thư mục!")
else:
    print(f"🕵️ Đang tính toán xác suất trên {len(face_files)} khung hình...\n")

    case_model.eval()
    for i, face_file in enumerate(face_files):
        img_path = os.path.join(face_folder, face_file)
        img = Image.open(img_path).convert('RGB')
        img_t = case_transform(img).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            outputs = case_model(img_t)
            # Chuyển Logits thành Xác suất (0 -> 1) bằng Softmax
            probs = F.softmax(outputs, dim=1)
            # Lấy xác suất của lớp FAKE
            fake_prob = probs[0][fake_idx].item() * 100
            list_fake_probs.append(fake_prob)

            # In ra log từng khung hình (tùy chọn)
            if (i+1) % 5 == 0 or i == 0:
                print(f"Frame {i+1}: {fake_prob:.2f}% Fake")

    # 4. TÍNH TRUNG BÌNH CỘNG TẤT CẢ CÁC KHUNG
    avg_fake_score = np.mean(list_fake_probs)

    # 5. IN BÁO CÁO TỔNG HỢP
    print("\n" + "="*45)
    print(f"📊 BÁO CÁO PHÂN TÍCH VIDEO CHI TIẾT")
    print("="*45)
    print(f"🔹 Tổng số khung hình quét: {len(list_fake_probs)}")
    print(f"🔹 Xác suất Fake thấp nhất: {np.min(list_fake_probs):.2f}%")
    print(f"🔹 Xác suất Fake cao nhất: {np.max(list_fake_probs):.2f}%")
    print("-" * 45)
    print(f"🚨 XÁC SUẤT DEEPFAKE TRUNG BÌNH: {avg_fake_score:.2f}%")
    print("-" * 45)

    # Kết luận dựa trên ngưỡng 50%
    if avg_fake_score > 50:
        print(f"👉 KẾT LUẬN: VIDEO CÓ DẤU HIỆU DEEPFAKE (% fake trung bình: {avg_fake_score:.2f}%")
    else:
        print(f"👉 KẾT LUẬN: VIDEO NGƯỜI THẬT (Độ tin cậy Real: {100-avg_fake_score:.2f}%")
    print("="*45)

    # 1. Tìm khung hình có xác suất Fake cao nhất để hiển thị
    max_idx = np.argmax(list_fake_probs)
    best_fake_score = list_fake_probs[max_idx]

    # 2. Tạo biến rep_img_np (Ảnh đại diện dạng Numpy)
    rep_img_path = os.path.join(face_folder, face_files[max_idx])
    rep_img_np = np.array(Image.open(rep_img_path).convert('RGB'))

    # 3. Đồng nhất tên biến trung bình (avg_fake_score -> avg_fake_percent)
    avg_fake_percent = avg_fake_score

    print(f"✅ Đã chuẩn bị xong dữ liệu hình ảnh từ khung hình: {face_files[max_idx]}")


## 5.5. Cài đặt thư viện LIME

In [ ]:
!pip install lime

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from lime import lime_image
from skimage.segmentation import mark_boundaries

## 5.6. Chạy LIME dùng chung cho mọi model

In [ ]:
# ==========================================
# 4. HIỂN THỊ KẾT QUẢ TỐI ƯU (ĐÃ SỬA LOGIC)
# ==========================================
fig, ax = plt.subplots(1, 3, figsize=(22, 8))

from lime import lime_image

# 1. Khởi tạo bộ giải thích LIME
explainer = lime_image.LimeImageExplainer()

# 2. Định nghĩa hàm dự đoán cho LIME
def batch_predict(images):
    case_model.eval()
    batch = torch.stack([case_transform(Image.fromarray(i)) for i in images], dim=0)
    batch = batch.to(DEVICE)
    logits = case_model(batch)
    probs = F.softmax(logits, dim=1)
    return probs.detach().cpu().numpy()

# 3. Chạy giải thích cho khung hình dự đoán deepfake cao nhất (Mất khoảng 1-2 phút)
print("🕵️ Đang phân tích vùng nghi vấn (LIME)... Vui lòng đợi...")
explanation = explainer.explain_instance(
    rep_img_np,
    batch_predict,
    top_labels=2,
    hide_color=0,
    num_samples=500 # Số mẫu càng cao càng chính xác nhưng càng chậm
)
print("✅ Đã tạo xong biến 'explanation'!")

# --- Cột 1: Ảnh gốc (Khung hình tiêu biểu nhất) ---
ax[0].imshow(rep_img_np)
# Chỉ hiện tên ảnh và thông báo đây là khung hình tiêu biểu
ax[0].set_title(f"KHUNG HÌNH ĐẠI DIỆN\n(Trích xuất từ video)", fontsize=14, color='blue')
ax[0].axis('off')

# --- Cột 2: Phân tích khung hình này (Dùng best_fake_score) ---
temp, mask = explanation.get_image_and_mask(fake_idx, positive_only=True, num_features=5, hide_rest=False)
ax[1].imshow(mark_boundaries(temp, mask, color=(1, 1, 0)))
# Hiển thị dự đoán RIÊNG cho khung hình này
ax[1].set_title(f"AI DỰ ĐOÁN KHUNG NÀY: {'FAKE' if best_fake_score > 50 else 'REAL'}\n(% fake: {best_fake_score:.2f}%)", fontsize=12)
ax[1].axis('off')

# --- Cột 3: Heatmap trọng số ( RdYlGn_r ) ---
dict_heatmap = dict(explanation.local_exp[fake_idx])
heatmap = np.vectorize(dict_heatmap.get)(explanation.segments)
im = ax[2].imshow(heatmap, cmap='RdYlGn_r')
ax[2].set_title("BẢN ĐỒ NHIỆT CHI TIẾT\n(Đỏ: Vùng nghi ngờ fake nhất)", fontsize=12)
ax[2].axis('off')
fig.colorbar(im, ax=ax[2], fraction=0.046, pad=0.04)

# --- DÒNG TỔNG KẾT: ĐÂY MỚI LÀ NƠI DÙNG AVG_FAKE_PERCENT ---
plt.figtext(0.5, 0.02,
            f"KẾT LUẬN TOÀN VIDEO: {'🚨 DEEPFAKE' if avg_fake_percent > 50 else '✅ NGƯỜI THẬT'} "
            f"(Xác suất trung bình: {avg_fake_percent:.2f}%)",
            ha="center", fontsize=18, fontweight='bold', bbox={"facecolor":"yellow", "alpha":0.5, "pad":10})

plt.tight_layout()
plt.show()

## 5.7. VIDEO DEMO


Định nghĩa một hàm Python mới có tên realtime_deepfake_detector để xử lý video theo thời gian thực. Hàm này sẽ nhận đường dẫn video đầu vào, mô hình deepfake đã huấn luyện (case_model), bộ phát hiện khuôn mặt (mtcnn), hàm biến đổi ảnh (case_transform), thiết bị (DEVICE), bộ giải thích LIME (explainer), và hàm dự đoán (batch_predict). Hàm sẽ đọc từng khung hình từ video, phát hiện khuôn mặt, thực hiện dự đoán, tạo giải thích LIME và vẽ trực tiếp lên khung hình. Video đầu ra sẽ bao gồm các khung hình đã được xử lý với các vùng LIME được highlight và xác suất deepfake cho từng khuôn mặt.

In [ ]:
import cv2
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
from skimage.segmentation import mark_boundaries
import imageio
import imageio_ffmpeg # Ensure this is available for imageio backend
import matplotlib.pyplot as plt # Import matplotlib for colormaps

def realtime_deepfake_detector(video_path, model, mtcnn, transform, device, explainer, predict_fn,
                               output_path='output.mp4', frames_to_process=None,
                               lime_num_samples=500, lime_num_features=5, fake_label_idx=0):
    """
    Detects deepfakes in real-time from a video, processes faces, applies LIME explanations,
    and saves the output video with visualizations.

    Args:
        video_path (str): Path to the input video file.
        model (torch.nn.Module): Trained deepfake detection model.
        mtcnn (facenet_pytorch.MTCNN): MTCNN face detection model.
        transform (torchvision.transforms.Compose): Image transformation for model input.
        device (torch.device): Device to run models on ('cuda' or 'cpu').
        explainer (lime_image.LimeImageExplainer): LIME image explainer.
        predict_fn (function): Prediction function for LIME.
        output_path (str): Path to save the output video.
        frames_to_process (int, optional): Maximum number of frames to process. None for all frames.
        lime_num_samples (int): Number of samples for LIME explanation.
        lime_num_features (int): Number of features for LIME explanation.
        fake_label_idx (int): Index of the 'FAKE' class in the model's output.
    """
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"Error: Could not open video file {video_path}")
        return

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"Input video properties: FPS={fps}, Resolution={frame_width}x{frame_height}, Total frames={total_frames}")

    # Use imageio writer (expects RGB images)
    writer = imageio.get_writer(output_path, fps=fps, codec='libx264', macro_block_size=None)

    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frames_to_process and frame_count >= frames_to_process:
            break

        # Convert OpenCV BGR frame to RGB for PIL and other processing
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_to_draw_on_rgb = frame_rgb.copy()

        # MTCNN expects PIL Image
        img_pil = Image.fromarray(frame_rgb)

        # Detect faces (boxes are [x1, y1, x2, y2])
        # mtcnn.detect returns (boxes, probabilities) or (None, None)
        boxes, _ = mtcnn.detect(img_pil)

        if boxes is not None:
            for box in boxes:
                x1, y1, x2, y2 = [int(b) for b in box]

                # Ensure bounding box coordinates are within frame dimensions
                x1 = max(0, x1)
                y1 = max(0, y1)
                x2 = min(frame_width, x2)
                y2 = min(frame_height, y2)

                # Extract face region from the original RGB frame
                face_img_orig_size_rgb = frame_rgb[y1:y2, x1:x2]
                if face_img_orig_size_rgb.size == 0: # Skip empty face crops
                    continue

                # Resize face for model and LIME. IMG_SIZE is 224 from previous cells.
                face_for_processing_pil = Image.fromarray(face_img_orig_size_rgb).resize((IMG_SIZE, IMG_SIZE))
                face_for_processing_np = np.array(face_for_processing_pil) # LIME expects numpy array

                # Get LIME explanation
                try:
                    explanation = explainer.explain_instance(
                        face_for_processing_np,
                        predict_fn,
                        top_labels=2,
                        hide_color=0,
                        num_samples=lime_num_samples
                    )

                    # Generate a colored heatmap overlay for blending
                    dict_heatmap_values = dict(explanation.local_exp[fake_label_idx])
                    raw_heatmap = np.vectorize(dict_heatmap_values.get)(explanation.segments)

                    # Normalize heatmap values for the colormap
                    if raw_heatmap.max() == raw_heatmap.min():
                        norm_heatmap = np.zeros_like(raw_heatmap)
                    else:
                        norm_heatmap = (raw_heatmap - raw_heatmap.min()) / (raw_heatmap.max() - raw_heatmap.min())

                    # Use matplotlib colormap (RdYlGn_r: Red-Yellow-Green, reversed so red means fake)
                    cmap = plt.get_cmap('RdYlGn_r')
                    colored_heatmap_overlay = cmap(norm_heatmap)[:,:,:3] # Get RGB, discard alpha
                    colored_heatmap_overlay_uint8 = (colored_heatmap_overlay * 255).astype(np.uint8)

                    # Resize to original face box dimensions
                    colored_heatmap_overlay_resized_pil = Image.fromarray(colored_heatmap_overlay_uint8).resize((x2 - x1, y2 - y1))
                    colored_heatmap_overlay_resized_np = np.array(colored_heatmap_overlay_resized_pil)

                    # Blend with the original face
                    face_part_float = face_img_orig_size_rgb.astype(np.float32)
                    overlay_float = colored_heatmap_overlay_resized_np.astype(np.float32)

                    alpha = 0.5 # Transparency for the LIME overlay
                    highlighted_face_rgb = cv2.addWeighted(face_part_float, 1 - alpha, overlay_float, alpha, 0)
                    highlighted_face_rgb = highlighted_face_rgb.astype(np.uint8)

                    # Overlay LIME result onto the main frame
                    frame_to_draw_on_rgb[y1:y2, x1:x2] = highlighted_face_rgb

                except Exception as e:
                    print(f"LIME explanation failed for a face in frame {frame_count}: {e}")
                    # If LIME fails, just draw the bounding box without explanation
                    cv2.rectangle(frame_to_draw_on_rgb, (x1, y1), (x2, y2), (0, 0, 255), 2) # Red bounding box

                # Model prediction for the face (using the processed 224x224 face)
                input_tensor = transform(face_for_processing_pil).unsqueeze(0).to(device)
                with torch.no_grad():
                    outputs = model(input_tensor)
                    probs = F.softmax(outputs, dim=1)
                    fake_prob = probs[0][fake_label_idx].item() * 100

                # Determine label text and color
                label_text = f"FAKE: {fake_prob:.2f}%" if fake_prob > 50 else f"REAL: {100-fake_prob:.2f}%"
                label_color = (255, 0, 0) if fake_prob > 50 else (0, 255, 0) # Red for FAKE, Green for REAL
                font_scale = 0.7
                font_thickness = 2

                # Get text size to draw a background rectangle
                (text_width, text_height), baseline = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, font_thickness)
                text_origin = (x1, y1 - 10)

                # Draw black background rectangle for the text
                cv2.rectangle(frame_to_draw_on_rgb, (text_origin[0], text_origin[1] - text_height - baseline),
                              (text_origin[0] + text_width, text_origin[1] + baseline), (0, 0, 0), -1)

                # Draw bounding box and label
                cv2.rectangle(frame_to_draw_on_rgb, (x1, y1), (x2, y2), label_color, 2)
                cv2.putText(frame_to_draw_on_rgb, label_text, text_origin,
                            cv2.FONT_HERSHEY_SIMPLEX, font_scale, label_color, font_thickness)
        else:
            # If no faces are detected in the current frame
            cv2.putText(frame_to_draw_on_rgb, "No face detected", (50, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

        # Write the processed frame (RGB) to the output video
        writer.append_data(frame_to_draw_on_rgb)
        frame_count += 1
        if frame_count % fps == 0: # Print status every second of video
            print(f"Processed {frame_count} frames...")

    cap.release()
    writer.close()
    print(f"Real-time deepfake detection complete. Output video saved to {output_path}")

In [ ]:
print("Starting real-time deepfake detection demo...")

# Ensure IMG_SIZE is correctly set for the model being used (EfficientNet here)
# It was set to 224 in cell a_J_RnRqwoDL and lzxdeKPOyGFf, so it's consistent.

# Call the realtime_deepfake_detector function
realtime_deepfake_detector(
    video_path=VIDEO_INPUT, # From previous cells, e.g., '/content/deepfake.mp4'
    model=case_model,       # Loaded model from previous cells
    mtcnn=mtcnn,            # MTCNN detector from previous cells
    transform=case_transform, # Transforms for model input
    device=DEVICE,          # Device ('cuda' or 'cpu')
    explainer=explainer,    # LIME explainer
    predict_fn=batch_predict, # Prediction function for LIME
    output_path='realtime_output.mp4', # Output video file name
    frames_to_process=500, # Process a limited number of frames for a quicker demo
    lime_num_samples=50,  # Reduce LIME samples for faster processing in real-time context
    lime_num_features=3,   # Reduce LIME features for faster processing
    fake_label_idx=fake_idx # Index for the 'FAKE' class
)

print("Real-time deepfake detection demo completed. Check 'realtime_output.mp4'.")